# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In Croissant, all entities are referenced by unique `@id` fields. This overview cell lists available record sets and their main fields for discovery and selection.


In [ ]:
# List all record sets defined by their @id
print("Available Record Sets (by @id):")
record_set_objs = list(dataset.iter_record_sets())
for rs in record_set_objs:
    print(f"- @id: {rs.id}\n  name: {rs.name}")
    # List fields (by @id) in this record set
    print("  Fields (@id -> name):")
    for field in rs.fields:
        print(f"    - {field.id} -> {field.name}")
    print("")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify all record set @ids
record_set_ids = [rs.id for rs in dataset.iter_record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print(f"No records found for record set @id: {record_set_id}")
    else:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Record set {record_set_id} loaded, columns: {df.columns.tolist()}")

# For demonstration, select the first non-empty record set for further analysis
main_record_set_id = None
for k in dataframes:
    if len(dataframes[k]) > 0:
        main_record_set_id = k
        break
if main_record_set_id is not None:
    print(f"Using record set {main_record_set_id} for EDA.")
    display(dataframes[main_record_set_id].head())
else:
    print("No non-empty record sets found.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we:
- Filter based on a numeric field (if found)
- Normalize this field
- Group by a categorical field (if found)


In [ ]:
import numpy as np

# Pick a numeric field (prioritize columns named like 'abundance', 'MW', 'count', 'coverage', etc.)
numeric_field = None
candidate_keywords = ['abundance', 'MW', 'count', 'coverage', 'peptide', 'score', 'id', 'mass', 'number']
df = dataframes[main_record_set_id]
for col in df.columns:
    for key in candidate_keywords:
        if key in col.lower():
            # Check if values can be numeric
            try:
                _ = pd.to_numeric(df[col], errors='coerce')
                numeric_field = col
                break
            except Exception:
                continue
    if numeric_field:
        break

if not numeric_field:
    # Fallback: pick the first field with numeric dtype
    numeric_cols = df.select_dtypes(include=np.number).columns
    if len(numeric_cols) > 0:
        numeric_field = numeric_cols[0]

if numeric_field:
    # Try to cast the column to numeric if not already
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.75)
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a categorical field
    # Look for candidate grouping fields (not the numeric)
    group_field = None
    for col in df.columns:
        if col == numeric_field:
            continue
        # Pick first column with low cardinality (could be 'description', 'sample', etc.)
        if df[col].nunique() <= min(10, len(df)//5):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_'+numeric_field)
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No group field with low cardinality found.")
else:
    print("No numeric field found for EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot the filtered numeric field's distribution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field].dropna(), bins=20, kde=True, color='skyblue')
    plt.title(f'Distribution of {numeric_field} (> {threshold})')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")


## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to:
- Discover record sets and field `@id`s in a Croissant dataset
- Dynamically select primary fields for analysis
- Filter, normalize, and group numeric data
- Visualize feature distributions

The FAIR² dataset provides structured, well-described experimental proteomic data suitable for downstream analysis. For targeted or more advanced analysis, consult the schema's full field definitions and use explicit selection by `@id` for reproducibility.
